# 02 SRG Grouping
Assigns edges to Shared Risk Groups and builds SRG graph objects.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import yaml
import pickle
from src.network.build import prepare_edge_df
from src.network.srg  import fill_srg_list, build_srg_graph, build_srg_dif_graph

# file paths
import yaml
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

# Reimport directories and load
NODE_DIR = config["paths"]["node_list"]
EDGE_DIR = config["paths"]["edge_list"]
IXP_DIR = config["paths"]["ixp_list"]
SRG_DIR = config["paths"]["srg_list"]
SITE_DIR = config["paths"]["site_list"]
GRAPH_DIR = config["paths"]["graph_folder"]
TABULAR_DIR = config["paths"]["tabular_folder"]

node_df = pd.read_csv(NODE_DIR)
edge_df = pd.read_csv(EDGE_DIR)
srg_city_df = pd.read_csv(SRG_DIR)
ixp_df = pd.read_csv(IXP_DIR)
site_df = pd.read_csv(SITE_DIR, comment='#')
edge_df = prepare_edge_df(edge_df)

# load graph object
with open(f"{GRAPH_DIR}/G_full.pickle", "rb") as f:
    G_full = pickle.load(f)

In [2]:
# Fill SRLG lists (both conservative and ideal)
srg_c_df, srg_i_df = fill_srg_list(edge_df, srg_city_df)

print('Conservative SRGs:', len(srg_c_df))
print('Ideal SRGs:', len(srg_i_df))
srg_c_df.head()

Conservative SRGs: 51
Ideal SRGs: 55


,srg_id,srg_type,u,v,edges
0,ADL_DRW_TER_1,rail_road,ADL_1,DRW_1,"[(ADL_DRW_1, ADL_1, DRW_1, telstra), (ADL_DRW_..."
1,ADL_SMAP_SUB_1,sub,ADL_1,SMAP_1,"[(MEL_ADL_2, MEL_1, ADL_1, telstra), (PER_ADL_..."
2,BNE_ROK_TER_1,rail_road,BNE_1,ROK_1,"[(BNE_DRW_1, BNE_1, DRW_1, telstra), (BNE_ROK_..."
3,BNE_ROK_TER_2,road,BNE_1,ROK_1,"[(BNE_ROK_2, BNE_1, ROK_1, aarnet)]"
4,BNE_ROK_TER_3,road,BNE_1,ROK_1,"[(BNE_TSV_1, BNE_1, TSV_1, optus), (BNE_ROK_3,..."


In [3]:
# Determine the type breakdown for the conservative set
print(srg_c_df['srg_type'].value_counts())

srg_type
road         20
sub          16
rail_road    15
Name: count, dtype: int64


In [4]:
# Build SRLG graph objects (conservative, ideal, and difference)
G_srg_c = build_srg_graph(srg_c_df, site_df)
G_srg_i = build_srg_graph(srg_i_df, site_df)
G_srg_dif = build_srg_dif_graph(srg_c_df, srg_i_df, site_df)

print('Conservative SRG graph:', G_srg_c)
print('Ideal SRG graph:', G_srg_i)
print('Difference graph:', G_srg_dif)

Conservative SRG graph: MultiGraph with 16 nodes and 50 edges
Ideal SRG graph: MultiGraph with 16 nodes and 54 edges
Difference graph: MultiGraph with 16 nodes and 4 edges


In [5]:
# Optional: save dataframes
import os
os.makedirs(TABULAR_DIR, exist_ok=True)
srg_c_df.to_csv(f'{TABULAR_DIR}/srg_c.csv', index=False)

In [6]:
# Optional: save graphs
import pickle
with open(f"{GRAPH_DIR}/G_srg_c.pickle", "wb") as f:
    pickle.dump(G_srg_c, f)